# EDA Notebook 6 — Student Enrollments Bronze
**Source**: `mini_project_grp6.bronze.enrollments_bronze`  
**Format**: CSV | ~1M rows | Daily delta with status changes  
**Goal**: Status distribution, progress patterns, overdue enrollments, dropout signals

In [0]:
from pyspark.sql import functions as F

TABLE = "mini_project_grp6.bronze.enrollments_bronze"
df    = spark.table(TABLE)
total = df.count()

print(f"Total rows: {total:,}")
df.printSchema()

In [0]:
# ============================================================
# CELL 2 — Null Audit
# ============================================================

null_df = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).toPandas().T.reset_index()
null_df.columns = ["column", "null_count"]
null_df["null_pct"] = (null_df["null_count"] / total * 100).round(2)
print(null_df.sort_values("null_count", ascending=False).to_string(index=False))

In [0]:
# ============================================================
# CELL 3 — Enrollment Status Distribution
# ============================================================
# ACTIVE | COMPLETED | DROPPED | PAUSED

display(
    df.groupBy("status")
      .agg(
          F.count("*").alias("count"),
          F.round(F.count("*") / total * 100, 2).alias("pct"),
          F.round(F.avg("current_progress_pct"), 2).alias("avg_progress_pct")
      )
      .orderBy("count", ascending=False)
)

In [0]:
# ============================================================
# CELL 4 — Progress % Distribution
# ============================================================

print("Progress % stats by status:")
display(
    df.groupBy("status")
      .agg(
          F.round(F.avg("current_progress_pct"), 2).alias("avg_pct"),
          F.round(F.expr("percentile_approx(current_progress_pct, 0.5)"), 2).alias("median_pct"),
          F.min("current_progress_pct").alias("min_pct"),
          F.max("current_progress_pct").alias("max_pct")
      )
)

# Flag invalid progress values (outside 0-100)
invalid_progress = df.filter(
    (F.col("current_progress_pct") < 0) |
    (F.col("current_progress_pct") > 100)
)
print(f"\nInvalid progress_pct (< 0 or > 100): {invalid_progress.count():,}")

In [0]:
# ============================================================
# CELL 5 — Dropout Risk Early Signals
# ============================================================
# Preview of what Silver risk classification will find

from datetime import datetime, timedelta

ref_date = datetime.now().strftime("%Y-%m-%d")

high_risk_preview = df.filter(
    (F.col("status") == "ACTIVE") &
    (F.datediff(F.lit(ref_date), F.col("last_accessed_date")) > 14) &
    (F.col("current_progress_pct") < 30)
)

medium_risk_preview = df.filter(
    (F.col("status") == "ACTIVE") &
    (
        (F.datediff(F.lit(ref_date), F.col("last_accessed_date")) > 7)
    )
)

print(f"Potential HIGH risk students  : {high_risk_preview.count():,}")
print(f"Inactive > 7 days (ACTIVE)   : {medium_risk_preview.count():,}")

In [0]:
# ============================================================
# CELL 6 — Overdue Enrollments
# ============================================================
# expected_completion_date has passed but status is still ACTIVE

overdue = df.filter(
    (F.col("status") == "ACTIVE") &
    (F.col("expected_completion_date") < F.lit(ref_date))
)

print(f"Overdue active enrollments: {overdue.count():,}")
display(
    overdue.select(
        "enrollment_id", "student_id", "course_id",
        "expected_completion_date", "current_progress_pct"
    ).limit(10)
)

In [0]:
# ============================================================
# CELL 7 — Days Since Last Access Distribution
# ============================================================

display(
    df.withColumn(
        "days_inactive",
        F.datediff(F.lit(ref_date), F.col("last_accessed_date"))
    )
    .withColumn("inactivity_bucket",
        F.when(F.col("days_inactive") <= 3, "0–3 days")
         .when(F.col("days_inactive") <= 7, "4–7 days")
         .when(F.col("days_inactive") <= 14, "8–14 days")
         .when(F.col("days_inactive") <= 30, "15–30 days")
         .otherwise("> 30 days")
    )
    .filter(F.col("status") == "ACTIVE")
    .groupBy("inactivity_bucket")
    .count()
    .orderBy("inactivity_bucket")
)

In [0]:
# ============================================================
# CELL 8 — Referential Integrity: course_id in catalog?
# ============================================================

catalog_df    = spark.table("mini_project_grp6.bronze.course_catalog_bronze")
valid_courses = catalog_df.select("course_id").distinct()

orphan_enrollments = df.join(
    valid_courses, on="course_id", how="left_anti"
)
print(f"Enrollments with no matching course_catalog entry: {orphan_enrollments.count():,}")

In [0]:
# ============================================================
# CELL 9 — Enrollment Volume Over Time
# ============================================================

display(
    df.withColumn("enrolled_month", F.date_format("enrolled_date", "yyyy-MM"))
      .groupBy("enrolled_month")
      .count()
      .orderBy("enrolled_month")
)

In [0]:
# ============================================================
# CELL 10 — Grand EDA Summary (run last)
# ============================================================

print("=" * 55)
print("ENROLLMENT BRONZE — EDA SUMMARY")
print("=" * 55)
print(f"  Total enrollments  : {total:,}")
print(f"  Unique students    : {df.select('student_id').distinct().count():,}")
print(f"  Unique courses     : {df.select('course_id').distinct().count():,}")
print(f"  Overdue ACTIVE     : (see Cell 6)")
print(f"  HIGH risk preview  : (see Cell 5)")
print("=" * 55)